# 11. Bias in inference and mitigation: loan approval

The loan table from [WorkshopIgualdad2025](https://github.com/rferper/WorkshopIgualdad2025) is where the workshop **changes the score** and follows how approval rates differ by gender. Same split, same short search, three objectives. `Gender` stays in `X` every time, so a rule may still say `Gender IS Male`. Only the reward changes.

| Where it acts | Idea | In `ebdai` |
| --- | --- | --- |
| Pre-processing | Drop `Gender` | not fitted here. Other columns can still track gender, and then no rule can say so |
| Pre-processing | Reweigh rows so group and label are independent (Kamiran and Calders 2012) | `reweigh_weights` + `weighted_mcc_loss` |
| In-processing | Charge the search for the selection-rate gap | `fairness_regularized_loss` |

Both fitted repairs use `customized_loss`, which replaces the score the search maximises. Read each gap next to its accuracy and MCC.

On [Google Colab](https://colab.research.google.com/github/HAISymbiosis/EBD-AI/blob/main/Demos/11_bias_loan_fairness.ipynb), run the setup cell first. If the next cell cannot import `ebdai`, restart the runtime and run every cell again.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HAISymbiosis/EBD-AI/blob/main/Demos/11_bias_loan_fairness.ipynb)


## Setup (Google Colab only)

Clones this repository and installs it. Skip it if `ebdai` is already installed (`pip install ebdai`, or `pip install -e .`).


In [ ]:
!git clone -q https://github.com/HAISymbiosis/EBD-AI.git
%cd EBD-AI
!pip install -q .


## The table, and the label gap

`load_loan_approval` drops incomplete rows and the loan id, recodes `Loan_Status` from `N`/`Y` to 0/1, and leaves 480 of 614 applications. Approval is 1. The sensitive column is `Gender`.

Before any model, 54 of 86 women are approved (62.8%) and 278 of 394 men are approved (70.6%), a difference of about eight points. Men are 394 of the 480 applicants. The test set holds 22 women and 137 men. Each of those women is about 1/22 of the women's rate, so a difference of two people is about nine points (2/22). On a group this small, read a modest difference together with the counts.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from ex_fuzzy import BaseFuzzyRulesClassifier, FUZZY_SETS, eval_tools
from ebdai import (
    features_and_target, load_loan_approval, outcome_rates_by_group,
    plot_outcome_rates, fairness_report, reweigh_weights,
    weighted_mcc_loss, fairness_regularized_loss,
    parse_printed_rules, winning_rules_by_group,
    plot_winning_rules_by_group,
)

frame, sensitive = load_loan_approval()
X, y = features_and_target(frame, 'Loan_Status')
print(X.head())
rates = outcome_rates_by_group(y, X[sensitive], positive_label=1)
print(rates)
plot_outcome_rates(rates, title='Loan approval rate by gender')

  Gender Married Dependents     Education Self_Employed  ApplicantIncome  \
1   Male     Yes          1      Graduate            No             4583   
2   Male     Yes          0      Graduate           Yes             3000   
3   Male     Yes          0  Not Graduate            No             2583   
4   Male      No          0      Graduate            No             6000   
5   Male     Yes          2      Graduate           Yes             5417   

   CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History  \
1             1508.0       128.0             360.0             1.0   
2                0.0        66.0             360.0             1.0   
3             2358.0       120.0             360.0             1.0   
4                0.0       141.0             360.0             1.0   
5             4196.0       267.0             360.0             1.0   

  Property_Area  
1         Rural  
2         Urban  
3         Urban  
4         Urban  
5         Urban  
    group    n

<Axes: title={'center': 'Loan approval rate by gender'}, xlabel='Group', ylabel='Positive rate'>

## Fit 1 — baseline: maximise MCC

`fit_rules` is the same fit three times; only the loss changes. `ds_mode=2` lets the search choose a weight per rule, so the decision score is firing strength × weight. This run passes no loss: the objective is MCC on the training rows, with no term for groups.

Stored run: 6 of 22 women approved and 79 of 137 men (27% and 58%). Demographic-parity gap 0.30, equalized-odds gap 0.31. Accuracy 94/159 (59%), below the 70% of approving everyone; MCC stays at 0.17. The search was never asked to avoid this gap.


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

def fit_rules(X_tr, y_tr, X_te, y_te, loss=None):
    clf = BaseFuzzyRulesClassifier(
        nRules=8, nAnts=3, fuzzy_type=FUZZY_SETS.t1,
        n_linguistic_variables=3, ds_mode=2, verbose=False,
        n_gen=6, pop_size=12, patience=3, random_state=42,
    )
    if loss is not None:
        clf.customized_loss(loss)
    clf.fit(X_tr, y_tr)
    report = eval_tools.eval_fuzzy_model(
        clf, X_tr, y_tr, X_te, y_te,
        plot_rules=False, print_rules=False, plot_partitions=False,
        return_rules=True, bootstrap_results_print=False,
    )
    return clf, report

clf, report = fit_rules(X_train, y_train, X_test, y_test)
y_pred = clf.predict(X_test)
table, gaps = fairness_report(y_test, y_pred, X_test[sensitive])
print('baseline')
print(table)
print(pd.Series(gaps))

------------
ACCURACY
Train performance: 0.5669781931464174
Test performance: 0.5911949685534591
------------
MATTHEW CORRCOEF
Train performance: 0.10374834639844925
Test performance: 0.16926832390925828
------------
baseline
    group    n  selection_rate       tpr       fpr       fnr       tnr
0    Male  137        0.576642  0.628866  0.450000  0.371134  0.550000
1  Female   22        0.272727  0.333333  0.142857  0.666667  0.857143
demographic_parity_difference    0.303915
demographic_parity_ratio         0.472957
equalized_odds_difference        0.307143
equalized_odds_ratio             0.317460
tpr_difference                   0.295533
fpr_difference                   0.307143
dtype: float64


## Fit 2 — reweighing the training rows (Kamiran and Calders, 2012)

Each training row gets a weight from its group `a` and its label `y`:

```
w(a, y) = P(Y = y) · P(A = a) / P(A = a, Y = y)
```

A group–label pair rarer than independence counts for more. In the weighted training sample, group and label are independent. `reweigh_weights` follows training-row order; pass those same rows to `fit`. Test metrics stay unweighted.

This changes the training reward. It does not fix the test gap. In the stored run the short search reverses the gap, and the difference grows: 17 of 22 women are approved (77%) and 45 of 137 men (33%), a difference of 0.44 in the other direction. Accuracy falls to 83/159 (52%); MCC is 0.15.


In [3]:
weights = reweigh_weights(y_train, X_train[sensitive])
clf_w, _ = fit_rules(
    X_train, y_train, X_test, y_test,
    loss=weighted_mcc_loss(weights),
)
table_w, gaps_w = fairness_report(
    y_test, clf_w.predict(X_test), X_test[sensitive]
)
print('reweighed')
print(table_w)
print(pd.Series(gaps_w))

------------
ACCURACY
Train performance: 0.49221183800623053
Test performance: 0.5220125786163522
------------
MATTHEW CORRCOEF
Train performance: 0.10050895214863548
Test performance: 0.15053717375108475
------------
reweighed
    group    n  selection_rate       tpr       fpr       fnr       tnr
0    Male  137        0.328467  0.381443  0.200000  0.618557  0.800000
1  Female   22        0.772727  0.800000  0.714286  0.200000  0.285714
demographic_parity_difference    0.444260
demographic_parity_ratio         0.425075
equalized_odds_difference        0.514286
equalized_odds_ratio             0.280000
tpr_difference                   0.418557
fpr_difference                   0.514286
dtype: float64


## Fit 3 — charging the genetic objective for the gap

Each candidate is scored as

```
score = MCC − λ × (highest selection rate − lowest selection rate)
```

with `lam=0.2`. The rates are computed on the training rows during the search. A 30-point gap costs 0.06, a large slice of an MCC around 0.10–0.25.

In the stored run the gap closes because both groups are approved less often: 4 of 22 women and 33 of 137 men (18% and 24%), demographic-parity difference 0.06. The TPR gap falls to about 7 points and the FPR gap to about 3. Accuracy is 68/159 (43%) and MCC is 0.10, against a 69% approval rate in the table itself.

Equal rates still leave open whether that shared rate is high or low. Across the three fits: a 30-point gap at 94/159 correct, a 44-point gap in the other direction at 83/159, and a 6-point gap at 68/159. The last lines show which rules the penalised search kept, and for whom they fire.


In [4]:
clf_f, report_f = fit_rules(
    X_train, y_train, X_test, y_test,
    loss=fairness_regularized_loss(X_train[sensitive], lam=0.2),
)
table_f, gaps_f = fairness_report(
    y_test, clf_f.predict(X_test), X_test[sensitive]
)
print('regularised')
print(table_f)
print(pd.Series(gaps_f))
counts = winning_rules_by_group(
    clf_f, X_test, X_test[sensitive],
    rule_texts=parse_printed_rules(report_f or ''),
)
plot_winning_rules_by_group(
    counts, title='Winning loan rules by gender (regularised fit)'
)

------------
ACCURACY
Train performance: 0.4174454828660436
Test performance: 0.4276729559748428
------------
MATTHEW CORRCOEF
Train performance: 0.09636798445458983
Test performance: 0.09580279131572642
------------
regularised
    group    n  selection_rate       tpr       fpr       fnr       tnr
0    Male  137        0.240876  0.268041  0.175000  0.731959  0.825000
1  Female   22        0.181818  0.200000  0.142857  0.800000  0.857143
demographic_parity_difference    0.059058
demographic_parity_ratio         0.754821
equalized_odds_difference        0.068041
equalized_odds_ratio             0.746154
tpr_difference                   0.068041
fpr_difference                   0.032143
dtype: float64


<Axes: title={'center': 'Winning loan rules by gender (regularised fit)'}, xlabel='Winning rule', ylabel='Share of group'>